Model Architecture, Mathematics, and Cloud Training Protocol

upload bi_lstm_mumbai_v1 and mumbai_coastal_vault.db over a folder named exactly as "Sentinel_V7" inside drive 
then copy the below code on a new notebook on google colab 
then execute the codes (make sure to select T4 or above GPU before starting the training)

The default training parameters are as follows:
"SEQ_LENGTH = 24  
EPOCHS = 150 # Increased epochs because the GPU is fast
TARGET_MAE = 0.0004" 

The parameter for training could be changed according to size or the intensity of application 
Points of Importance :
1) At the core of Sentinel V7 is a deep **Bi-LSTM** network. Standard LSTMs process time-series data sequentially from past to present, updating a hidden state. However, urban flooding—especially in coastal zones subject to tidal locks—exhibits complex temporal dependencies where downstream choke points influence upstream water pooling.
2) To capture this "temporal shockwave" effect, the Bi-LSTM processes the 8-factor telemetry tensor in two directions simultaneously:
2.1) **Forward Pass:** Computes the hidden state  considering historical data (past rainfall accumulation).
2.2)  **Backward Pass:** Computes the hidden state  capturing the contextual future state (e.g., an incoming high tide that will halt drainage).
3) The final hidden state representation is the concatenation of both directions, providing the network with absolute temporal context before passing it through the dense layers for the final depth prediction



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
# Point Python to your Sentinel folder so it can find your model file
sys.path.append('/content/drive/MyDrive/Sentinel_V7')

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import sqlite3
import pandas as pd
import numpy as np
import os
import time
from sklearn.preprocessing import MinMaxScaler

# Imports from your Google Drive
from bi_lstm_mumbai_v1 import SentinelMumbaiPINN, PINN_Loss

# --- CLOUD CONFIGURATION ---
# Pointing directly to your Google Drive Vault
DB_PATH = '/content/drive/MyDrive/Sentinel_V7/mumbai_coastal_vault.db'
MODEL_SAVE_PATH = '/content/drive/MyDrive/Sentinel_V7/sentinel_mumbai_v1_cloud.pt'
SEQ_LENGTH = 24  
EPOCHS = 150 # Increased epochs because the GPU is fast
TARGET_MAE = 0.0004

# Hardware auto-detection: Will lock onto the NVIDIA T4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- GPU HYPER-OPTIMIZATIONS ---
PHYSICAL_BATCH_SIZE = 256    # Massive increase for GPU VRAM 
ACCUMULATION_STEPS = 4       # Effective batch size of 1024

class MumbaiCoastalDataset(Dataset):
    def __init__(self, db_path, seq_length):
        print("[SYS] Extracting Tactical Telemetry (Monsoon Filtering Active)...")
        conn = sqlite3.connect(db_path)
        
        query = """
            SELECT 
                t.precip_mm_hr, t.cumulative_rain_24h, t.tidal_height_m, 
                w.avg_elevation_m, w.impermeability_index, w.drainage_capacity_m3_s, w.distance_to_coast_m,
                0.0 AS temp_placeholder, 0.0 AS soil_placeholder, 
                t.actual_flood_depth_m
            FROM telemetry_mesh t
            JOIN ward_topography w ON t.ward_id = w.ward_id
            WHERE t.precip_mm_hr > 0.0 OR t.tidal_height_m > 3.0
            ORDER BY t.ward_id, t.timestamp
        """
        df = pd.read_sql_query(query, conn)
        conn.close()

        features = df.iloc[:, :-1].values
        targets = df.iloc[:, -1].values
        
        print(f"[SYS] Filtered Dataset Size: {len(df)} critical risk records.")

        self.scaler_x = MinMaxScaler()
        self.scaler_y = MinMaxScaler()
        
        features_scaled = self.scaler_x.fit_transform(features)
        targets_scaled = self.scaler_y.fit_transform(targets.reshape(-1, 1))

        num_samples = len(features_scaled) - seq_length
        idx = np.arange(seq_length)[None, :] + np.arange(num_samples)[:, None]
        
        self.X = torch.FloatTensor(features_scaled[idx])
        self.y = torch.FloatTensor(targets_scaled[seq_length:])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def initiate_cloud_forge():
    print(f"=== INITIATING SENTINEL V7 CLOUD FORGE ===")
    print(f"[HW] Active Compute Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"[HW] GPU Type: {torch.cuda.get_device_name(0)}")
        print(f"[HW] Precision Opt: Native CUDA Automatic Mixed Precision (AMP)")

    dataset = MumbaiCoastalDataset(DB_PATH, SEQ_LENGTH)
    
    # num_workers=2 utilizes Linux background threading for faster data loading
    dataloader = DataLoader(dataset, batch_size=PHYSICAL_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
    
    total_batches = len(dataloader)
    print(f"[OK] Dataloader ready. Total Batches/Epoch: {total_batches}")

    model = SentinelMumbaiPINN(input_dim=9, hidden_dim=512, num_layers=6).to(DEVICE)
    
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"[SYS] Locating previous cloud checkpoint...")
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True))
        print(f"[OK] Weights loaded successfully. Resuming optimization.")

    pinn_loss_fn = PINN_Loss(alpha=1.0, beta=10.0) 
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # GPU Scaler for Mixed Precision Math
    scaler = torch.amp.GradScaler()

    print("\n[SYS] Commencing GPU Deep Learning Protocol...")
    best_loss = float('inf')
    forge_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        total_data_loss, total_phys_loss, total_mae = 0.0, 0.0, 0.0
        epoch_start = time.time()
        
        optimizer.zero_grad()

        for batch_idx, (X_batch, y_batch) in enumerate(dataloader):
            # Move data from RAM to GPU VRAM
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            
            # FAST GPU MATH: Calculate in Half-Precision
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                predictions = model(X_batch)
                
                rain_intensity = X_batch[:, -1, 0:1] 
                tidal_height = X_batch[:, -1, 2:3]
                
                loss, d_loss, p_loss = pinn_loss_fn(predictions, y_batch, rain_intensity, tidal_height)
                loss = loss / ACCUMULATION_STEPS

            # Scale gradients back up to prevent underflow
            scaler.scale(loss).backward()
            
            if ((batch_idx + 1) % ACCUMULATION_STEPS == 0) or (batch_idx + 1 == total_batches):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() 
            
            total_data_loss += d_loss.item()
            total_phys_loss += p_loss.item()
            total_mae += torch.mean(torch.abs(predictions.float() - y_batch.view_as(predictions))).item()

        scheduler.step()

        avg_d_loss = total_data_loss / total_batches
        avg_p_loss = total_phys_loss / total_batches
        avg_mae = total_mae / total_batches
        epoch_time = time.time() - epoch_start
        
        print(f">> EPOCH [{epoch+1:03d}/{EPOCHS}] COMPLETED IN {epoch_time:.1f}s | MAE: {avg_mae:.6f} | Data Loss: {avg_d_loss:.6f}")

        if avg_mae < best_loss:
            best_loss = avg_mae
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"   [+] New Minimum. Saved to Google Drive.")
            
        if avg_mae <= TARGET_MAE:
            print(f"\n[!!!] TARGET PRECISION ACHIEVED ({TARGET_MAE} MAE).")
            break

    elapsed = (time.time() - forge_start) / 60
    print(f"\n=== CLOUD FORGE COMPLETE ===")
    print(f"Total GPU Training Time : {elapsed:.2f} minutes")
    print(f"Peak Precision (MAE): {best_loss:.6f}")

if __name__ == "__main__":
    initiate_cloud_forge()